## Approach - 1: Extraction through MA Legislature API

> Massachussets Legislature API provides structured access to state legislative data which inclubes information about bills, laws, committees, legislators, hearings, sessions, votes, and related legislative actions. Using this API, we can extract details such as bill texts and summaries, sponsor information, committee membership, and records of how legislators voted on different measures.

In [ ]:
# Below block focuses on extracting bills raised in the current year (2025-26 with generalCourtNumber=194)
# Here we get all the bills raised for a specific year. We have a unique generalCourtNumber assigned for every consecutive 2 years.

import requests
import pandas as pd

generalCourtNumber = 194 # for current year (2025-26)

url='https://malegislature.gov/api/GeneralCourts/194/Documents'
# Make a GET request to the API endpoint
resp = requests.get(url)
if resp.status_code == 404:
    print("No data found.")

resp.raise_for_status()
data = resp.json()
res=[]

# Extract relevant bill details
for i in data: 
    res.append({
                'GeneralCourtNumber': generalCourtNumber,
                'BillNumber':i.get('BillNumber'),
               'Title': i.get('Title')})

# Store the extracted bill details in a DataFrame
Bill_details= pd.DataFrame(res)
display(Bill_details)

,GeneralCourtNumber,BillNumber,Title
0,194,H3444,An Act relative to s-license compliance
1,194,H1283,An Act relative to real estate disclosure forms
2,194,H1399,An Act relative to an individual Medicare mark...
3,194,H1282,An Act relative to the Massachusetts Uniform C...
4,194,H1932,An Act relative to the uniform voidable transa...
...,...,...,...
9145,194,S2759,Revenue - Extension Order
9146,194,None,An Act directing the city of Boston Police Dep...
9147,194,H4794,"Order relative to extending until Wednesday, M..."
9148,194,None,STEM Pipeline Fund FY26 Transfer Schedule


- This block extracts Senate roll call votes for each recorded roll call votes.
- RollCallNumber is unique for every roll call session recorded. (Note- A specific bill can have multiple roll call numbers assigned based on number of voting occurred for the bill before it is finally approved)


In [ ]:
import requests
import pandas as pd

generalCourtNumber = 194 # for current year (2025-26)
branch = "Senate" # senate roll call votes
rollCallNumber = 56 # session for which votes are recorded

# Extract roll calls for given session
def get_rollcalls(generalCourtNumber, branch, rollCallNumber):
    url = f"https://malegislature.gov/api/GeneralCourts/{generalCourtNumber}/Branches/{branch}/RollCalls/{rollCallNumber}"
    resp = requests.get(url)
    if resp.status_code == 404:
        print("No data found.")
        return None
    resp.raise_for_status()
    data = resp.json()
    return data


data = get_rollcalls(generalCourtNumber, branch, rollCallNumber)

# Extract member name from API for roll call votes 
# Create a dataframe storing final data
res=[]
for i in data.keys(): 
    votes = data.get(i, [])  
    if not votes:            
        continue
    if i not in [ 'Yeas','Nays','Absent']: 
        continue
    if len(data[i])==0: 
        continue
    x=0
    for j in data[i]:
        x+=1
        member_url= j['Details']
        # Handle case when unable to fetch member code or url
        if not member_url:
            res.append({
                        'RollCallNumber': rollCallNumber,
                        'Name': 'NA', 
                        'MemberCode': j.get('MemberCode', 'NA'), 
                        'Vote': i})
            continue
        resp_mem = requests.get(member_url)
        # Handle case when unable to fetch member code or url
        if resp_mem.status_code == 404:
            res.append({
                        'RollCallNumber': rollCallNumber,
                        'Name':'NA', 
                        'MemberCode': j.get('MemberCode', 'NA'),
                        'Vote':i})
            continue
        resp_mem.raise_for_status()
        data_mem = resp_mem.json()
        res.append({
                    'RollCallNumber': rollCallNumber,
                    'Name':data_mem['Name'], 
                    'MemberCode':j['MemberCode'], 
                    'Vote':i})

senate_votes = pd.DataFrame(res)
print(senate_votes)


    RollCallNumber                      Name MemberCode  Vote
0               56           Kelly A. Dooner       KAD0  Yeas
1               56             Bruce E. Tarr       BET0  Yeas
2               56       Patrick M. O'Connor        PMO  Yeas
3               56           Ryan C. Fattman       RCF0  Yeas
4               56           Peter J. Durant       PJD0  Yeas
5               56        Dylan A. Fernandes       DAF0  Nays
6               56  William J. Driscoll, Jr.       WJD0  Nays
7               56              Paul W. Mark       PWM0  Nays
8               56           Pavel M. Payano       PMP0  Nays
9               56          Robyn K. Kennedy       RKK0  Nays
10              56         Jacob R. Oliveira       JRO0  Nays
11              56               Liz Miranda       L M0  Nays
12              56         Barry R. Finegold       BRF0  Nays
13              56       Brendan P. Crighton       BPC0  Nays
14              56       Cynthia Stone Creem       CSC0  Nays
15      

In [6]:
# Extract bill roll call mapping for a specific bill

def get_bill_rollcall_map(generalCourtNumber, documentNumber):
    url = f'https://malegislature.gov/api/GeneralCourts/{generalCourtNumber}/Documents/{documentNumber}/RollCalls'
    resp = requests.get(url)
    if resp.status_code == 404:
        print("No data found.")
    resp.raise_for_status()
    data = resp.json()

    # From extracted data, get roll call numbers for the specific bill
    res=[]
    for i in data: 
        res.append({
                    'GeneralCourtNumber': generalCourtNumber,
                    'BillNumber': documentNumber,
                    'RollCallNumber': i.get('RollCallNumber')
                })
    Bill_rollcall_map= pd.DataFrame(res)
    return Bill_rollcall_map

bill_rollcall_map=get_bill_rollcall_map(194, 'S2538')
display(bill_rollcall_map)


,GeneralCourtNumber,BillNumber,RollCallNumber
0,194,S2538,55
1,194,S2538,56
2,194,S2538,57


In [7]:
#final join 
result1= pd.merge(Bill_details, bill_rollcall_map, how='inner', on='BillNumber', suffixes=('', '_drop'))
result2= pd.merge(result1, senate_votes, how='inner', on='RollCallNumber', suffixes=('', '_drop'))
denorm_data = result2[[c for c in result2.columns if not c.endswith('_drop')]]
denorm_data.loc[:, 'Vote']= denorm_data['Vote'].replace({'Yeas':'Y', 'Nays':'N'})
display(denorm_data.head())

,GeneralCourtNumber,BillNumber,Title,RollCallNumber,Name,MemberCode,Vote
0,194,S2538,An Act strengthening health care protections i...,56,Kelly A. Dooner,KAD0,Y
1,194,S2538,An Act strengthening health care protections i...,56,Bruce E. Tarr,BET0,Y
2,194,S2538,An Act strengthening health care protections i...,56,Patrick M. O'Connor,PMO,Y
3,194,S2538,An Act strengthening health care protections i...,56,Ryan C. Fattman,RCF0,Y
4,194,S2538,An Act strengthening health care protections i...,56,Peter J. Durant,PJD0,Y
